# Evaluation Pipeline

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Add src to path for imports
sys.path.insert(0, os.path.abspath("../../src"))

# --- Sample data for demonstration ---
np.random.seed(42)
n = 500
dates = pd.bdate_range("2020-01-01", periods=n)

# Simulate adjusted-scale true values (e.g., sqrt of realized vol)
y_true = np.abs(np.random.randn(n) * 0.5 + 1.0)
# Forecasts with some noise
forecasts = y_true + np.random.randn(n) * 0.15
# Baselines (e.g., long-run variance estimate)
baselines = np.abs(np.random.randn(n) * 0.01 + 0.02)

print(f"Sample size: {n}")
print(f"y_true range: [{y_true.min():.3f}, {y_true.max():.3f}]")
print(f"forecasts range: [{forecasts.min():.3f}, {forecasts.max():.3f}]")

## Duan Smearing

In [ ]:
# Duan (1983) smearing estimator
#
# When we model on a transformed scale (sqrt, log), naive retransformation
# (just squaring the forecast) is biased because E[g(X)] != g(E[X]).
#
# The smearing correction adds back the average squared residual:
#   smear = mean((y_true - forecasts)^2)
#   pred_raw = (forecasts^2 + smear) * baseline
#   true_raw = (y_true^2) * baseline
#
# This provides an unbiased estimate on the original (raw) scale.

smear = np.mean((y_true - forecasts) ** 2)
pred_raw = (forecasts**2 + smear) * baselines
true_raw = (y_true**2) * baselines

print(f"Smearing factor: {smear:.6f}")
print(f"Naive pred_raw mean (no smear): {np.mean(forecasts**2 * baselines):.6f}")
print(f"Corrected pred_raw mean:        {np.mean(pred_raw):.6f}")
print(f"True raw mean:                  {np.mean(true_raw):.6f}")

fig, ax = plt.subplots(1, 1, figsize=(10, 4))
ax.plot(dates[:100], true_raw[:100], label="true_raw", alpha=0.8)
ax.plot(dates[:100], pred_raw[:100], label="pred_raw (smeared)", alpha=0.8)
ax.set_title("Duan Smearing: Raw-Scale Predictions vs Truth")
ax.legend()
ax.set_ylabel("Raw volatility")
plt.tight_layout()
plt.show()

## QLIKE Loss

In [ ]:
# QLIKE loss (Patton, 2011):
#   QLIKE_i = (true_raw / pred_raw) - log(true_raw / pred_raw) - 1
#
# Properties:
# - Always >= 0 (equals 0 when prediction == truth)
# - Asymmetric: penalizes under-prediction more heavily than over-prediction
# - Robust loss for variance forecasting (consistent for MSE-type ranking)

mask = (true_raw > 0) & (pred_raw > 0)
ratio = true_raw[mask] / pred_raw[mask]
qlike_vals = ratio - np.log(ratio) - 1.0

print(f"Mean QLIKE: {np.mean(qlike_vals):.6f}")
print(f"Median QLIKE: {np.median(qlike_vals):.6f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(qlike_vals, bins=50, edgecolor="black", alpha=0.7)
axes[0].axvline(np.mean(qlike_vals), color="red", linestyle="--", label=f"mean={np.mean(qlike_vals):.4f}")
axes[0].set_title("QLIKE Distribution")
axes[0].set_xlabel("QLIKE")
axes[0].legend()

# Show asymmetry: QLIKE as function of ratio
r = np.linspace(0.2, 5.0, 200)
q = r - np.log(r) - 1.0
axes[1].plot(r, q)
axes[1].axvline(1.0, color="gray", linestyle=":", alpha=0.5)
axes[1].set_title("QLIKE vs true/pred ratio")
axes[1].set_xlabel("true_raw / pred_raw")
axes[1].set_ylabel("QLIKE")
axes[1].annotate("under-prediction\n(higher penalty)", xy=(3.5, 1.5), fontsize=9)
axes[1].annotate("over-prediction", xy=(0.3, 0.5), fontsize=9)

plt.tight_layout()
plt.show()

## MSE and MAE

In [ ]:
# Standard metrics on adjusted scale
errors = y_true - forecasts
mse = np.mean(errors**2)
mae = np.mean(np.abs(errors))
print(f"MSE (adjusted): {mse:.6f}")
print(f"MAE (adjusted): {mae:.6f}")

# Winsorized variants: clip errors to [5th, 95th] percentile
lo = np.percentile(errors, 5)
hi = np.percentile(errors, 95)
w_errors = np.clip(errors, lo, hi)
w_mse = np.mean(w_errors**2)
w_mae = np.mean(np.abs(w_errors))
print(f"\nWinsorized MSE: {w_mse:.6f}")
print(f"Winsorized MAE: {w_mae:.6f}")
print(f"\nWinsorization bounds: [{lo:.4f}, {hi:.4f}]")
print(f"Samples clipped: {np.sum((errors < lo) | (errors > hi))} / {len(errors)}")

In [ ]:
%%writefile ../../src/evaluation.py
"""Evaluation metrics and Duan smearing for volatility forecasts.

Standalone module — no imports from core/ or projects/.
"""

import numpy as np
import pandas as pd


def winsorize_array(arr: np.ndarray, lower_q: float = 0.05, upper_q: float = 0.95) -> np.ndarray:
    """Clip values to [lower_q, upper_q] percentile bounds."""
    lo = np.percentile(arr, lower_q * 100)
    hi = np.percentile(arr, upper_q * 100)
    return np.clip(arr, lo, hi)


def apply_duan_smearing(
    forecasts: np.ndarray,
    y_true: np.ndarray,
    baselines: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """Apply Duan smearing correction to convert adjusted-scale forecasts to raw scale.

    Parameters
    ----------
    forecasts : array-like
        Model predictions on adjusted (sqrt / log) scale.
    y_true : array-like
        True values on adjusted scale.
    baselines : array-like
        Baseline volatility used to scale back to raw units.

    Returns
    -------
    pred_raw : np.ndarray
        Smearing-corrected predictions on raw scale.
    true_raw : np.ndarray
        True values on raw scale.
    """
    forecasts = np.asarray(forecasts, dtype=np.float64)
    y_true = np.asarray(y_true, dtype=np.float64)
    baselines = np.asarray(baselines, dtype=np.float64)

    smear = np.mean((y_true - forecasts) ** 2)
    pred_raw = (forecasts ** 2 + smear) * baselines
    true_raw = (y_true ** 2) * baselines
    return pred_raw, true_raw


def build_results_dataframe(
    forecasts: np.ndarray,
    y_subset: np.ndarray,
    dates_subset: np.ndarray,
    baselines_subset: np.ndarray,
    horizon: int = 1,
) -> pd.DataFrame:
    """Build a tidy results DataFrame with adjusted and raw-scale columns.

    Parameters
    ----------
    forecasts : array-like
        Model predictions (adjusted scale).
    y_subset : array-like
        True targets (adjusted scale).
    dates_subset : array-like
        Corresponding dates.
    baselines_subset : array-like
        Baseline volatility for raw-scale conversion.
    horizon : int
        Forecast horizon label.

    Returns
    -------
    pd.DataFrame
        Columns: date, horizon, true_adj, pred_adj, true_raw, pred_raw.
    """
    forecasts = np.asarray(forecasts, dtype=np.float64)
    y_subset = np.asarray(y_subset, dtype=np.float64)
    baselines_subset = np.asarray(baselines_subset, dtype=np.float64)

    pred_raw, true_raw = apply_duan_smearing(forecasts, y_subset, baselines_subset)

    return pd.DataFrame({
        "date": dates_subset,
        "horizon": horizon,
        "true_adj": y_subset,
        "pred_adj": forecasts,
        "true_raw": true_raw,
        "pred_raw": pred_raw,
    })


def calculate_metrics(df: pd.DataFrame) -> dict:
    """Compute evaluation metrics from a results DataFrame.

    Parameters
    ----------
    df : pd.DataFrame
        Must contain columns: true_adj, pred_adj, true_raw, pred_raw.

    Returns
    -------
    dict
        Keys: mse, mae, w_mse, w_mae, qlike, w_qlike, n_samples.
    """
    true_adj = df["true_adj"].values
    pred_adj = df["pred_adj"].values
    true_raw = df["true_raw"].values
    pred_raw = df["pred_raw"].values

    # --- Adjusted-scale errors ---
    errors = true_adj - pred_adj
    mse = float(np.mean(errors ** 2))
    mae = float(np.mean(np.abs(errors)))

    # Winsorized variants
    w_errors = winsorize_array(errors)
    w_mse = float(np.mean(w_errors ** 2))
    w_mae = float(np.mean(np.abs(w_errors)))

    # --- QLIKE (raw scale) ---
    mask = (true_raw > 0) & (pred_raw > 0)
    if mask.sum() > 0:
        ratio = true_raw[mask] / pred_raw[mask]
        qlike_vals = ratio - np.log(ratio) - 1.0
        qlike = float(np.mean(qlike_vals))
        w_qlike = float(np.mean(winsorize_array(qlike_vals)))
    else:
        qlike = np.nan
        w_qlike = np.nan

    return {
        "mse": mse,
        "mae": mae,
        "w_mse": w_mse,
        "w_mae": w_mae,
        "qlike": qlike,
        "w_qlike": w_qlike,
        "n_samples": len(df),
    }